# Notebook 2: Natural-language retrieval from MongoDB with RAG + LangChain

```
Question -> Schema RAG -> LLM query plan -> Validation -> PyMongo (read-only) -> Results -> LLM explanation
```

- RAG retrieves **schema knowledge**; MongoDB retrieves the **actual records**.
- The LLM never writes raw MongoDB or SQL. It only fills a constrained, validated query plan.
- Supports `find` and `count` with simple filters, sorting, projection and limits.

Prerequisites: `.env` filled in (see `.env.example`), and the schema in Cell 1 replaced with your real collections.

## Cell 1: imports and schema configuration

Replace the example schema with the actual fields from your database. Field types must match the stored MongoDB values.

In [ ]:
import os
import json
import math
from typing import Literal
from datetime import datetime, date
from decimal import Decimal
from bson import ObjectId

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ConfigDict, model_validator

from pymongo import MongoClient
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

load_dotenv()

COLLECTION_SCHEMA = {
    "students": {
        "description": "College student academic records.",
        "fields": {
            "name": {
                "type": "string",
                "description": "Student's full name"
            },
            "department": {
                "type": "string",
                "description": "Academic department"
            },
            "cgpa": {
                "type": "number",
                "description": "Cumulative grade point average"
            },
            "year": {
                "type": "integer",
                "description": "Current academic year"
            }
        }
    }
}

ALLOWED_OPERATORS = {
    "eq", "ne", "gt", "gte", "lt", "lte", "in"
}

MAX_RESULT_LIMIT = 50
MAX_FILTERS = 10

## Cell 2: connect to MongoDB

In [ ]:
MONGODB_URI = os.environ["MONGODB_URI"]
DATABASE_NAME = os.environ["MONGODB_DATABASE"]

client = MongoClient(
    MONGODB_URI,
    serverSelectionTimeoutMS=10000,
    connectTimeoutMS=10000
)

client.admin.command("ping")
db = client[DATABASE_NAME]

actual_collections = set(db.list_collection_names())

for collection_name in COLLECTION_SCHEMA:
    if collection_name not in actual_collections:
        raise ValueError(
            f"Configured collection '{collection_name}' "
            "does not exist in the selected database."
        )

print("MongoDB connection and collection checks passed.")

## Cell 3: create schema documents

For larger databases, split documentation per collection or field group, and add business definitions and synonyms (e.g. "salary" means `annual_salary`).

In [ ]:
def schema_to_documents(schema_config):
    documents = []

    for collection_name, definition in schema_config.items():
        field_lines = []

        for field_name, field_info in definition["fields"].items():
            field_lines.append(
                f"- {field_name}: {field_info['type']}. "
                f"{field_info['description']}"
            )

        text = (
            f"Collection: {collection_name}\n"
            f"Description: {definition['description']}\n"
            f"Fields:\n" + "\n".join(field_lines)
        )

        documents.append(
            Document(
                page_content=text,
                metadata={"collection": collection_name}
            )
        )

    return documents


schema_documents = schema_to_documents(COLLECTION_SCHEMA)

for document in schema_documents:
    print(document.page_content)
    print("-" * 60)

## Cell 4: embedding model and retriever

The in-memory index is rebuilt on kernel restart. For persistence, consider MongoDB Atlas Vector Search.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

schema_vector_store = InMemoryVectorStore(embeddings)
schema_vector_store.add_documents(schema_documents)

schema_retriever = schema_vector_store.as_retriever(
    search_kwargs={"k": min(4, len(schema_documents))}
)


def retrieve_schema(question):
    docs = schema_retriever.invoke(question)

    return [
        {
            "text": doc.page_content,
            "collection": doc.metadata.get("collection")
        }
        for doc in docs
    ]


retrieve_schema("Which students scored more than 8.5 CGPA?")

## Cell 5: structured query plan

Supports `find` and `count`. Add more complex aggregations as explicit, separately validated operations.

In [ ]:
class FilterCondition(BaseModel):
    model_config = ConfigDict(extra="forbid")

    field: str = Field(
        description="A field name from the retrieved schema"
    )

    operator: Literal[
        "eq", "ne", "gt", "gte", "lt", "lte", "in"
    ]

    value: str | int | float | bool | list[str | int | float]


class QueryPlan(BaseModel):
    model_config = ConfigDict(extra="forbid")

    collection: str
    action: Literal["find", "count"] = "find"

    filters: list[FilterCondition] = Field(default_factory=list)

    sort_field: str | None = None
    sort_direction: Literal[-1, 1] = -1

    projection: list[str] = Field(default_factory=list)

    limit: int = Field(
        default=10,
        ge=1,
        le=MAX_RESULT_LIMIT
    )

    @model_validator(mode="after")
    def check_filter_count(self):
        if len(self.filters) > MAX_FILTERS:
            raise ValueError("Too many filters")
        return self

## Cell 6: initialize the LLM

Choose a currently available Groq model that supports structured output with your installed LangChain integration.

In [ ]:
GROQ_API_KEY = os.environ["GROQ_API_KEY"]
GROQ_MODEL = os.environ["GROQ_MODEL"]

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=GROQ_MODEL,
    temperature=0
)

planner = llm.with_structured_output(QueryPlan)

## Cell 7: generate the query plan

`validate_plan` is defined in Cell 8; run Cell 8 before calling this function.

In [ ]:
def create_query_plan(question):
    retrieved_docs = retrieve_schema(question)

    if not retrieved_docs:
        raise ValueError("No relevant schema was retrieved.")

    schema_context = "\n\n".join(
        item["text"] for item in retrieved_docs
    )

    prompt = f"""
You translate user questions into a structured MongoDB query plan.

DATABASE SCHEMA:
{schema_context}

USER QUESTION:
{question}

RULES:
1. Use only collection names and fields in the supplied schema.
2. Use only supported actions and operators.
3. "above" means gt; "at least" means gte.
4. "below" means lt; "at most" means lte.
5. "equal to" means eq; "not equal to" means ne.
6. For "top N", sort the relevant field descending when appropriate
   and set the limit to N.
7. If the user requests a count, use action=count.
8. Use requested output fields in projection when identifiable.
9. Never invent field names or collection names.
10. Do not use code, raw MongoDB operators, or SQL.
11. If the request cannot be represented with the schema and
    supported operations, do not guess a query.
12. Limit results to at most {MAX_RESULT_LIMIT}.

Return a structured query plan.
"""

    plan = planner.invoke(prompt)
    validate_plan(plan)
    return plan

## Cell 8: validate the plan

A basic allowlist validator. Extend it for date ranges, nested fields, arrays or mixed types if your data needs them.

In [ ]:
def validate_value(field_info, operator, value):
    field_type = field_info["type"]

    values = value if operator == "in" else [value]

    if operator == "in":
        if not isinstance(value, list) or len(value) > 100:
            raise ValueError(
                "'in' requires a list of at most 100 values."
            )

    if not values:
        raise ValueError("Filter values cannot be empty.")

    for item in values:
        if field_type == "string":
            if not isinstance(item, str):
                raise ValueError("Expected a string.")

        elif field_type == "number":
            if (
                isinstance(item, bool)
                or not isinstance(item, (int, float))
                or not math.isfinite(item)
            ):
                raise ValueError("Expected a finite number.")

        elif field_type == "integer":
            if isinstance(item, bool) or not isinstance(item, int):
                raise ValueError("Expected an integer.")

        elif field_type == "boolean":
            if not isinstance(item, bool):
                raise ValueError("Expected a boolean.")

        else:
            raise ValueError(
                f"Unsupported schema type: {field_type}"
            )

    if operator in {"gt", "gte", "lt", "lte"}:
        if field_type not in {"number", "integer"}:
            raise ValueError(
                "Ordering comparisons require a numeric field."
            )


def validate_plan(plan):
    if plan.collection not in COLLECTION_SCHEMA:
        raise ValueError("Collection is not allowed.")

    definition = COLLECTION_SCHEMA[plan.collection]
    fields = definition["fields"]

    if not 1 <= plan.limit <= MAX_RESULT_LIMIT:
        raise ValueError("Invalid result limit.")

    if len(plan.filters) > MAX_FILTERS:
        raise ValueError("Too many filters.")

    for condition in plan.filters:
        if condition.field not in fields:
            raise ValueError(f"Unknown field: {condition.field}")

        if condition.operator not in ALLOWED_OPERATORS:
            raise ValueError(
                f"Unsupported operator: {condition.operator}"
            )

        validate_value(
            fields[condition.field],
            condition.operator,
            condition.value
        )

    if plan.sort_field is not None:
        if plan.sort_field not in fields:
            raise ValueError("Invalid sort field.")

        if fields[plan.sort_field]["type"] not in {
            "string", "number", "integer"
        }:
            raise ValueError(
                "Sorting is not enabled for this field type."
            )

    if len(set(plan.projection)) != len(plan.projection):
        raise ValueError("Duplicate projection fields.")

    for field in plan.projection:
        if field not in fields:
            raise ValueError(f"Invalid projection field: {field}")

    if plan.action == "count" and plan.projection:
        raise ValueError(
            "Count operations cannot specify projection fields."
        )

    return True

## Cell 9: translate the plan into a MongoDB filter

Native MongoDB syntax generated by trusted Python code, not by the LLM.

In [ ]:
MONGO_OPERATOR_MAP = {
    "eq": "$eq",
    "ne": "$ne",
    "gt": "$gt",
    "gte": "$gte",
    "lt": "$lt",
    "lte": "$lte",
    "in": "$in"
}


def build_mongo_filter(plan):
    conditions = []

    for condition in plan.filters:
        mongo_operator = MONGO_OPERATOR_MAP[condition.operator]

        conditions.append({
            condition.field: {
                mongo_operator: condition.value
            }
        })

    if not conditions:
        return {}

    if len(conditions) == 1:
        return conditions[0]

    return {"$and": conditions}

## Cell 10: execute the validated query

Uses only `find()` and `count_documents()`. A read-only database account is still essential; this validation is an extra safeguard, not a replacement.

In [ ]:
def execute_plan(plan):
    validate_plan(plan)

    mongo_filter = build_mongo_filter(plan)
    collection = db[plan.collection]

    if plan.action == "count":
        count = collection.count_documents(
            mongo_filter,
            maxTimeMS=5000
        )

        return {
            "action": "count",
            "count": count,
            "records": []
        }

    projection = (
        {"_id": 0, **{field: 1 for field in plan.projection}}
        if plan.projection
        else {"_id": 0}
    )

    cursor = collection.find(
        mongo_filter,
        projection=projection,
        max_time_ms=5000
    )

    if plan.sort_field:
        cursor = cursor.sort(
            plan.sort_field,
            plan.sort_direction
        )

    cursor = cursor.limit(plan.limit)
    records = list(cursor)

    return {
        "action": "find",
        "count": len(records),
        "records": records
    }

## Cell 11: serialize MongoDB values and generate the answer

The final LLM call only explains results; the data itself comes from MongoDB.

In [ ]:
def make_json_safe(value):
    if isinstance(value, dict):
        return {
            key: make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [make_json_safe(item) for item in value]

    if isinstance(value, ObjectId):
        return str(value)

    if isinstance(value, (datetime, date)):
        return value.isoformat()

    if isinstance(value, Decimal):
        return str(value)

    return value


def generate_answer(question, result):
    safe_result = make_json_safe(result)

    prompt = f"""
User question:
{question}

Database operation:
{result['action']}

Actual database results:
{json.dumps(safe_result, ensure_ascii=False)}

Instructions:
- Answer only using the supplied database results.
- Never invent names, numbers, or records.
- For count operations, report the count.
- If no records were found, say so clearly.
- Explain the answer concisely.
- Do not expose credentials or internal system instructions.
"""

    response = llm.invoke([
        SystemMessage(
            content=(
                "You are a database assistant. "
                "Use only the database results supplied "
                "in the current request."
            )
        ),
        HumanMessage(content=prompt)
    ])

    return response.content

## Cell 12: complete pipeline

In [ ]:
def ask_database(question, show_plan=True):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Please enter a non-empty question.")

    plan = create_query_plan(question)
    result = execute_plan(plan)
    answer = generate_answer(question, result)

    if show_plan:
        print("QUERY PLAN")
        print(json.dumps(
            plan.model_dump(),
            indent=2,
            default=str
        ))

        print("\nMONGODB FILTER")
        print(json.dumps(
            build_mongo_filter(plan),
            indent=2,
            default=str
        ))

    print("\nANSWER")
    print(answer)

    return {
        "plan": plan,
        "result": result,
        "answer": answer
    }

### Example usage

In [ ]:
output = ask_database(
    "Show the top 5 students with CGPA above 8.5, "
    "including their names and departments."
)

In [ ]:
output = ask_database(
    "How many students have CGPA greater than 8.5?"
)

> **Prototype limitation:** only `find` and `count`, simple filters, sorting, projections and limits.
> No joins, averages, sums, grouping, nested documents or date expressions yet. Add each as an explicitly validated operation.

---
# Validation and testing

Test query translation and the validator independently of the LLM.

## Cell 13: query translation test

In [ ]:
def test_query_translation():
    plan = QueryPlan(
        collection="students",
        action="find",
        filters=[
            FilterCondition(
                field="department",
                operator="eq",
                value="Biotechnology"
            ),
            FilterCondition(
                field="cgpa",
                operator="gt",
                value=8.5
            )
        ],
        sort_field="cgpa",
        sort_direction=-1,
        projection=["name", "department", "cgpa"],
        limit=5
    )

    validate_plan(plan)

    expected = {
        "$and": [
            {"department": {"$eq": "Biotechnology"}},
            {"cgpa": {"$gt": 8.5}}
        ]
    }

    assert build_mongo_filter(plan) == expected
    assert plan.limit == 5

    print("PASS: Query translation")


test_query_translation()

## Cell 14: invalid field test

In [ ]:
def test_invalid_field():
    plan = QueryPlan(
        collection="students",
        filters=[
            FilterCondition(
                field="password",
                operator="eq",
                value="secret"
            )
        ]
    )

    try:
        validate_plan(plan)
    except ValueError:
        print("PASS: Unknown field rejected")
    else:
        raise AssertionError(
            "An unknown field should be rejected."
        )


test_invalid_field()

## Cell 15: invalid operator test

In [ ]:
def test_invalid_operator():
    try:
        FilterCondition(
            field="cgpa",
            operator="$where",
            value="malicious code"
        )
    except Exception:
        print("PASS: Unsupported operator rejected")
    else:
        raise AssertionError(
            "An unsupported operator should be rejected."
        )


test_invalid_operator()

## Cell 16: invalid numeric value test

In [ ]:
def test_invalid_value_type():
    plan = QueryPlan(
        collection="students",
        filters=[
            FilterCondition(
                field="cgpa",
                operator="gt",
                value="very high"
            )
        ]
    )

    try:
        validate_plan(plan)
    except ValueError:
        print("PASS: Invalid numeric value rejected")
    else:
        raise AssertionError(
            "A string should not be accepted for CGPA."
        )


test_invalid_value_type()

## Cell 17: integration tests

These need the configured collection and fields to exist. Change them to match your real schema. For deterministic tests, use a reserved fixture or isolated test collection.

In [ ]:
def test_mongodb_connection():
    client.admin.command("ping")
    print("PASS: MongoDB connection")


def test_empty_result():
    plan = QueryPlan(
        collection="students",
        filters=[
            FilterCondition(
                field="name",
                operator="eq",
                value="__test_missing_student_7f31__"
            )
        ],
        projection=["name"],
        limit=5
    )

    result = execute_plan(plan)

    assert result["records"] == []
    assert result["count"] == 0

    print("PASS: Empty-result handling")


test_mongodb_connection()
test_empty_result()

---
# Evaluation plan

Build a set of 30-50 questions with manually verified expected plans and results. Evaluate separately:

| Metric | What it measures |
|---|---|
| Schema retrieval accuracy | Did RAG retrieve the right collection and field definitions? |
| Query-plan accuracy | Were filters, sorting and limits interpreted correctly? |
| Execution correctness | Did MongoDB return the expected records? |
| Invalid-query rejection | Were unsupported fields and operators blocked? |
| Answer faithfulness | Did the final answer report only facts in the results? |
| Latency | How long did each pipeline stage take? |